# 03 - SHAP Analysis: Explaining the Heart Disease Model

## What is SHAP?
**SHAP (SHapley Additive exPlanations)** assigns each feature a "Shapley value" — the average marginal contribution of that feature across all possible subsets of features.

## Visualisations We Will Generate
| Plot | What it Shows |
|------|--------------|
| Summary (beeswarm) | Global feature importance + direction of effect for every sample |
| Bar chart | Mean absolute SHAP value per feature (global ranking) |
| Waterfall | Individual-level explanation — why did THIS patient get THIS prediction? |

**Key Question:** Which features push the model toward or away from a heart-disease prediction?

In [0]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

# Load pre-trained model and preprocessed data
with open('/tmp/preprocessed_data.pkl', 'rb') as f:
    data = pickle.load(f)

with open('/tmp/models.pkl', 'rb') as f:
    models = pickle.load(f)

X_train = data['X_train']
X_test  = data['X_test']
y_test  = data['y_test']
feature_names = data['feature_names']

best_model = models['xgb']   # XGBoost had the best ROC-AUC

# Initialise SHAP explainer (tree-based, fast)
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test)

print(f"SHAP values shape: {shap_values.shape}")
print("Explainer ready — TreeExplainer used for XGBoost.")

In [0]:
# SHAP Summary Plot (beeswarm) — global overview
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test, feature_names=feature_names, show=False)
plt.title("SHAP Summary Plot (Beeswarm) — XGBoost Heart Disease Model", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

# SHAP Bar Chart — mean absolute SHAP values
plt.figure(figsize=(9, 5))
shap.summary_plot(shap_values, X_test, feature_names=feature_names, plot_type='bar', show=False)
plt.title("SHAP Feature Importance (Mean |SHAP Value|)", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()

In [0]:
# Waterfall plots for two individual predictions
shap_explanation = shap.Explanation(
    values=shap_values,
    base_values=np.full(len(X_test), explainer.expected_value),
    data=X_test,
    feature_names=feature_names
)

# Case 1: Predicted as high risk (disease)
high_risk_idx = int(np.where(best_model.predict(X_test) == 1)[0][0])
plt.figure(figsize=(10, 5))
shap.waterfall_plot(shap_explanation[high_risk_idx], show=False, max_display=10)
plt.title(f"Waterfall Plot — High Risk Patient (idx {high_risk_idx})", fontsize=12)
plt.tight_layout()
plt.savefig('/tmp/shap_waterfall_high.png', dpi=150, bbox_inches='tight')
plt.show()

# Case 2: Predicted as low risk (no disease)
low_risk_idx = int(np.where(best_model.predict(X_test) == 0)[0][0])
plt.figure(figsize=(10, 5))
shap.waterfall_plot(shap_explanation[low_risk_idx], show=False, max_display=10)
plt.title(f"Waterfall Plot — Low Risk Patient (idx {low_risk_idx})", fontsize=12)
plt.tight_layout()
plt.savefig('/tmp/shap_waterfall_low.png', dpi=150, bbox_inches='tight')
plt.show()

## SHAP Findings Summary

| Rank | Feature | Typical SHAP Direction | Clinical Meaning |
|------|---------|------------------------|-----------------|
| 1 | cp (chest pain type) | Positive (non-typical = lower risk) | Asymptomatic chest pain strongly linked to CAD |
| 2 | thalach (max heart rate) | Negative (high HR = lower risk) | Higher max HR during stress test indicates healthy heart |
| 3 | ca (major vessels coloured) | Positive (more vessels = higher risk) | Vessel blockages detected by fluoroscopy |
| 4 | oldpeak (ST depression) | Positive (high = higher risk) | Ischemia marker — ST depression indicates reduced blood flow |
| 5 | thal (thalassemia type) | Positive (reversible defect = higher risk) | Reversible defect in thal scan indicates intermittent ischemia |

**Key Insight:** The top SHAP features align well with established cardiology knowledge, giving us confidence that the model has learned clinically meaningful patterns rather than noise.